In [25]:
import numpy as np
import pandas as pd
from deap import base, creator, tools, algorithms
from sklearn.preprocessing import StandardScaler

In [26]:
# Carga del dataset Wine Quality
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
df = pd.read_csv(url, sep=';')
X = df.drop('quality', axis=1)
y = df['quality'].values.reshape(-1, 1)

In [27]:
print("Resumen del Dataset Wine Quality:")
print(df.describe())

Resumen del Dataset Wine Quality:
       fixed acidity  volatile acidity  citric acid  residual sugar  \
count    1599.000000       1599.000000  1599.000000     1599.000000   
mean        8.319637          0.527821     0.270976        2.538806   
std         1.741096          0.179060     0.194801        1.409928   
min         4.600000          0.120000     0.000000        0.900000   
25%         7.100000          0.390000     0.090000        1.900000   
50%         7.900000          0.520000     0.260000        2.200000   
75%         9.200000          0.640000     0.420000        2.600000   
max        15.900000          1.580000     1.000000       15.500000   

         chlorides  free sulfur dioxide  total sulfur dioxide      density  \
count  1599.000000          1599.000000           1599.000000  1599.000000   
mean      0.087467            15.874922             46.467792     0.996747   
std       0.047065            10.460157             32.895324     0.001887   
min       0.01

In [28]:
# Normalización de variables
scaler = StandardScaler()
X_norm = scaler.fit_transform(X)

In [29]:
# Definición de parámetros
n_pesos = (11 * 8) + (8 * 1) # 11 inputs * 8 neuronas + 8 neuronas * 1 output

In [30]:
def forward_pass(pesos):
    pesos_arr = np.array(pesos)
    W1 = pesos_arr[:88].reshape(11, 8)
    W2 = pesos_arr[88:].reshape(8, 1)
    
    # Red neuronal simple: Capa oculta con tanh
    h = np.tanh(np.dot(X_norm, W1)) 
    return np.dot(h, W2)

In [ ]:
def evalFitness(individual):
    pred = forward_pass(individual)
    mse = np.mean((pred - y)**2) # Error Cuadrático Medio. El AG buscará minimizar este valor.
    return (mse,)

In [32]:
# Crear tipos para el AG
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

/opt/anaconda3/lib/python3.13/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMin' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/opt/anaconda3/lib/python3.13/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "


In [33]:
toolbox = base.Toolbox()
toolbox.register("attr_float", np.random.uniform, -1, 1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_float, n_pesos)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("mate", tools.cxBlend, alpha=0.5)
toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=0.2, indpb=0.1)
toolbox.register("select", tools.selTournament, tournsize=3)
toolbox.register("evaluate", evalFitness)

In [34]:
# Parámetros del AG
pop = toolbox.population(n=100)

In [35]:
# Evolución durante 50 generaciones
algorithms.eaSimple(pop, toolbox, cxpb=0.5, mutpb=0.2, ngen=50, verbose=True)

gen	nevals
0  	100   
1  	66    
2  	61    
3  	67    
4  	71    
5  	58    
6  	59    
7  	58    
8  	58    
9  	58    
10 	61    
11 	66    
12 	53    
13 	60    
14 	65    
15 	67    
16 	57    
17 	68    
18 	62    
19 	65    
20 	54    
21 	55    
22 	57    
23 	51    
24 	74    
25 	74    
26 	49    
27 	71    
28 	57    
29 	47    
30 	51    
31 	62    
32 	55    
33 	58    
34 	50    
35 	69    
36 	74    
37 	60    
38 	55    
39 	61    
40 	52    
41 	58    
42 	63    
43 	67    
44 	52    
45 	64    
46 	61    
47 	54    
48 	57    
49 	53    
50 	68    


([[-0.402336288088882,
   0.7430279112617271,
   2.3368981584436446,
   -0.07503539948224265,
   0.6035136374203722,
   -0.3123570776722879,
   -0.6100128660540698,
   -0.5696535934847498,
   -0.9115594835070271,
   -0.7142680282594666,
   1.7587109648887487,
   -0.3640041101052937,
   0.1914192779421686,
   0.8936747975831161,
   0.25396304619955534,
   -0.16497942158757703,
   -5.287948070217822,
   0.017084843064358174,
   -0.8996069065566796,
   1.941625950312885,
   -0.20587093810154553,
   0.8851390292574156,
   0.9813718504030486,
   -0.45092699671742476,
   -0.22384399400462335,
   0.15258272272210596,
   1.2876026599434365,
   -0.023556264435196227,
   9.651789918648099,
   0.3205330686823481,
   -2.4148352393829633,
   6.876355417345122,
   -1.2639672465972427,
   -0.3453457709319948,
   -0.6436231268929271,
   -0.11778572691278977,
   -0.04056174983849631,
   -0.3550711829402618,
   -0.4736027043835813,
   4.961162082583671,
   0.6422307783734595,
   0.5940131792627025,
   -

In [36]:
# Obtener el mejor individuo
best_ind = tools.selBest(pop, 1)[0]
print(f"\nMejor MSE alcanzado por AG: {best_ind.fitness.values[0]:.4f}")


Mejor MSE alcanzado por AG: 22.0347
